<a href="https://colab.research.google.com/github/marycasa20/environmental-satellite-analysis/blob/main/01_sentinel2_rgb_reciente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Mosaico Sentinel-2 en color verdadero

**Objetivo:** Construir una imagen reciente y libre de nubes para inspección territorial.

**Datos:** COPERNICUS/S2_SR_HARMONIZED.

**Relevancia para política ambiental y social:** Línea base visual para ordenamiento territorial, conservación y monitoreo.

**Limitaciones:** Un mosaico mediano puede ocultar eventos breves; validar fechas y nubes.


In [ ]:
# Instalar dependencias en Google Colab
!pip -q install earthengine-api geemap

import ee
import geemap
import datetime

ee.Authenticate()
ee.Initialize(project="TU_PROYECTO_GEE")

# Área de estudio de ejemplo: entorno de Chachapoyas, Amazonas, Perú
# Reemplázala por un polígono, activo de Earth Engine o coordenadas propias.
aoi = ee.Geometry.Point([-77.87, -6.23]).buffer(30000)

Map = geemap.Map()
Map.centerObject(aoi, 9)


In [ ]:
def mask_s2_sr(image):
    scl = image.select("SCL")
    clear = (
        scl.neq(3)   # sombra
        .And(scl.neq(8))  # nube media
        .And(scl.neq(9))  # nube alta
        .And(scl.neq(10)) # cirrus
        .And(scl.neq(11)) # nieve/hielo
    )
    return image.updateMask(clear).divide(10000).copyProperties(
        image, ["system:time_start"]
    )

def s2_composite(start, end):
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
        .map(mask_s2_sr)
        .median()
        .clip(aoi)
    )


In [ ]:
end = ee.Date(datetime.date.today().isoformat())
start = end.advance(-6, "month")
image = s2_composite(start, end)

Map.addLayer(image, {"bands":["B4","B3","B2"],"min":0,"max":0.3}, "Sentinel-2 RGB")
Map.addLayer(aoi, {}, "Área de estudio", False)
Map


In [ ]:
import geemap
from pathlib import Path

output_dir = Path("/content/maps")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "01_sentinel2_rgb_chachapoyas.png"

geemap.get_image_thumbnail(
    sentinel_composite,
    output=str(output_file),
    vis_params={
        "bands": ["B4", "B3", "B2"],
        "min": 0,
        "max": 0.30,
        "gamma": 1.2,
    },
    dimensions=1200,
    region=aoi,
    format="png",
)

print(f"Mapa guardado en: {output_file}")